# Indian-language audio (m4a) to text with Sarvam AI

Outputs the **detected language**, the **transcript in the original language**, and an **English translation**.

**Setup (once):**
1. Get an API key at https://dashboard.sarvam.ai
2. In Colab, click the key icon (Secrets) in the left sidebar, add a secret named `SARVAM_API_KEY`, and enable notebook access.

The REST endpoint only handles short clips (~30 s), so this notebook splits long audio into chunks and joins the results.

In [ ]:
!pip -q install pydub requests
# ffmpeg is preinstalled on Colab (needed by pydub for m4a)

In [ ]:
import io, requests
from google.colab import userdata, files
from pydub import AudioSegment

API_KEY = userdata.get('SARVAM_API_KEY')
URL = 'https://api.sarvam.ai/speech-to-text'

# 'unknown' = auto-detect language. Or set e.g. 'hi-IN', 'ta-IN', 'te-IN', 'kn-IN', 'bn-IN', 'mr-IN'.
LANGUAGE = 'unknown'
MODEL = 'saaras:v3'
CHUNK_SECONDS = 25

In [ ]:
uploaded = files.upload()  # pick your .m4a file
path = next(iter(uploaded))
audio = AudioSegment.from_file(path)
print(f'{path}: {len(audio)/1000:.1f} s')

In [ ]:
def call_sarvam(seg, mode):
    buf = io.BytesIO()
    seg.set_channels(1).set_frame_rate(16000).export(buf, format='wav')
    buf.seek(0)
    r = requests.post(
        URL,
        headers={'api-subscription-key': API_KEY},
        files={'file': ('chunk.wav', buf, 'audio/wav')},
        data={'model': MODEL, 'mode': mode, 'language_code': LANGUAGE},
        timeout=120,
    )
    r.raise_for_status()
    return r.json()

step = CHUNK_SECONDS * 1000
native, english, langs = [], [], []
for i, start in enumerate(range(0, len(audio), step)):
    seg = audio[start:start + step]
    t = call_sarvam(seg, 'transcribe')   # original language
    e = call_sarvam(seg, 'translate')    # English
    native.append(t.get('transcript', '').strip())
    english.append(e.get('transcript', '').strip())
    langs.append(t.get('language_code'))
    print(f'chunk {i+1} done ({t.get("language_code")})')

native_text = ' '.join(p for p in native if p)
english_text = ' '.join(p for p in english if p)
detected = sorted({l for l in langs if l})

print('\nDetected language(s):', ', '.join(detected))
print('\n--- Original ---\n' + native_text)
print('\n--- English ---\n' + english_text)

out = path.rsplit('.', 1)[0] + '.txt'
with open(out, 'w', encoding='utf-8') as f:
    f.write('Detected language(s): ' + ', '.join(detected) + '\n\n')
    f.write('--- Original ---\n' + native_text + '\n\n')
    f.write('--- English ---\n' + english_text + '\n')
    if 'clinical' in globals():  # only when the optional clinical cell was run
        f.write('\n--- Clinical interpretation ---\n' + clinical.get('interpretation', '') + '\n')
        f.write('\n--- Clinical note (English) ---\n' + clinical.get('note_english', '') + '\n')
        if clinical.get('note_native'):
            f.write('\n--- Clinical note (original language) ---\n' + clinical['note_native'] + '\n')
        f.write('\n--- Verification required ---\n')
        for item in clinical.get('verify', []):
            f.write('! ' + str(item) + '\n')
        f.write('\nAI-generated draft. A clinician must review it before use.\n')
files.download(out)

In [ ]:
import json, re

ENRICH_PROVIDER = 'openrouter'    # 'openrouter' or 'gemini'
ENRICH_MODEL = 'openrouter/free'  # e.g. 'openrouter/free'; with 'gemini' use e.g. 'gemini-3.5-flash'

LANG_NAMES = {'te': 'Telugu', 'hi': 'Hindi', 'ta': 'Tamil', 'kn': 'Kannada', 'ml': 'Malayalam',
              'bn': 'Bengali', 'mr': 'Marathi', 'gu': 'Gujarati', 'pa': 'Punjabi', 'en': 'English'}
# Spoken language name; '' or 'English' gives an English-only note. Defaults to the detected language.
SPOKEN_LANGUAGE = LANG_NAMES.get(detected[0].split('-')[0], '') if detected else ''


def clinical_prompt(transcript, english, language):
    native = None if language in (None, '', 'English') else language
    p = "You are a clinical scribe. Below is a clinician's dictated note"
    if native:
        p += f' (spoken in {native}, possibly mixed with English)'
    p += ('. Reply with JSON only, in the form {"interpretation": "...", '
          '"note_english": "...", "note_native": "...", "verify": ["..."]}.\n'
          '- "interpretation": a structured reading of what was said, as plain-text '
          'lines such as "Patient:", "Complaint:", "Findings:", "Diagnosis:", '
          '"Medications:", "Plan:". Omit a line if it was not mentioned.\n'
          '- "note_english": a clean clinical note in English (SOAP style where it '
          'fits), using standard medical terminology.\n'
          '- "note_native": the same clinical note written in ')
    p += native or 'the spoken language'
    p += (' (keep drug names, doses and standard medical terms in English)' if native
          else ', or an empty string if the note was in English')
    p += ('.\n- "verify": short items the clinician must double-check: drug names or '
          'doses that may be misheard or misspelled, ambiguous abbreviations '
          '(e.g. "SOS"), numbers, laterality, allergies, and anything unclear. '
          'Use an empty list only if nothing is doubtful.\n'
          'Use only what was said. Never invent findings, doses, diagnoses or '
          'names; if something is missing, leave it out.\n\n'
          f'Transcript:\n{transcript}')
    if english:
        p += f'\n\nEnglish translation:\n{english}'
    return p


def complete_json(prompt):
    if ENRICH_PROVIDER == 'openrouter':
        r = requests.post(
            'https://openrouter.ai/api/v1/chat/completions',
            headers={'Authorization': 'Bearer ' + userdata.get('OPENROUTER_API_KEY')},
            json={'model': ENRICH_MODEL, 'messages': [{'role': 'user', 'content': prompt}]},
            timeout=180,
        )
        r.raise_for_status()
        return r.json()['choices'][0]['message']['content']
    r = requests.post(
        f'https://generativelanguage.googleapis.com/v1beta/models/{ENRICH_MODEL}:generateContent',
        headers={'x-goog-api-key': userdata.get('GEMINI_API_KEY')},
        json={'contents': [{'parts': [{'text': prompt}]}],
              'generationConfig': {'responseMimeType': 'application/json'}},
        timeout=180,
    )
    r.raise_for_status()
    return ''.join(p.get('text', '') for p in r.json()['candidates'][0]['content']['parts'])


def extract_json(reply):
    reply = re.sub(r'<think>[\s\S]*?</think>', '', reply).strip()
    return json.loads(reply[reply.index('{'):reply.rindex('}') + 1])


clinical = extract_json(complete_json(clinical_prompt(native_text, english_text, SPOKEN_LANGUAGE)))
print(f'Model: {ENRICH_PROVIDER} / {ENRICH_MODEL}')
print('\n--- Clinical interpretation ---\n' + clinical.get('interpretation', ''))
print('\n--- Clinical note (English) ---\n' + clinical.get('note_english', ''))
if clinical.get('note_native'):
    print(f'\n--- Clinical note ({SPOKEN_LANGUAGE or "original language"}) ---\n' + clinical['note_native'])
print('\n--- Verification required ---')
for item in clinical.get('verify', []):
    print('  ! ' + str(item))
print('\nAI-generated draft. A clinician must review it before use.')

In [ ]:
out = path.rsplit('.', 1)[0] + '.txt'
with open(out, 'w', encoding='utf-8') as f:
    f.write('Detected language(s): ' + ', '.join(detected) + '\n\n')
    f.write('--- Original ---\n' + native_text + '\n\n')
    f.write('--- English ---\n' + english_text + '\n')
files.download(out)